#Celebal Technologies Summer Internship 2026
## Bronze Layer: Raw Data Ingestion
### FMCG Data Consolidation & Analytics Platform


**Objective:** Ingest raw data from multiple source systems (Company A & Company B) into Bronze Delta tables preserving original data for traceability and auditing.

**Sources:**
- Company A: Superstore Sales Dataset (CSV)
- Company B: Legacy FMCG System (Simulated)

In [0]:
# BRONZE LAYER: Raw Data Ingestion
# Medallion Architecture - Layer 1

from pyspark.sql.functions import lit, current_timestamp

print("=" * 60)
print("BRONZE LAYER: Data Ingestion Started")
print("=" * 60)
print(f"Spark Version: {spark.version}")

# Create Bronze database
spark.sql("CREATE DATABASE IF NOT EXISTS bronze_fmcg")
print("bronze_fmcg database ready!")

BRONZE LAYER: Data Ingestion Started
Spark Version: 4.1.0
bronze_fmcg database ready!


In [0]:
# Source 1: Company A - Superstore CSV

df_company_a = spark.read.csv(
    "/Workspace/Users/vasupainulyself@gmail.com/Drafts/Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

# Add metadata columns
df_company_a = df_company_a \
    .withColumn("source_company", lit("Company_A")) \
    .withColumn("source_system", lit("Superstore_ERP")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("batch_id", lit("BATCH_001"))

print("=== Company A: Raw Data Loaded ===")
print(f"Rows: {df_company_a.count()} | Columns: {len(df_company_a.columns)}")
df_company_a.show(5)

=== Company A: Raw Data Loaded ===
Rows: 9994 | Columns: 25
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+--------------+--------------+--------------------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|source_company| source_system| ingestion_timestamp| batch_id|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+--------------+--------------+------

In [0]:

# Source 2: Company B - Legacy FMCG System

from pyspark.sql.types import *

company_b_data = [
    ("B-ORD-1001", "2018-01-05", "B-CUST-001", "Raj Enterprises", "FMCG", "North India", "Delhi", "Soap & Detergent", "Surf Excel 1kg", 2500.0, 50, 0.05, 312.5),
    ("B-ORD-1002", "2018-01-10", "B-CUST-002", "Mumbai Traders", "Wholesale", "West India", "Mumbai", "Beverages", "Coca Cola 2L", 1800.0, 30, 0.10, 180.0),
    ("B-ORD-1003", "2018-01-15", "B-CUST-003", "Chennai Retail", "Retail", "South India", "Chennai", "Snacks", "Lays Classic 100g", 3200.0, 80, 0.0, 640.0),
    ("B-ORD-1004", "2018-01-20", "B-CUST-004", "Kolkata Mart", "FMCG", "East India", "Kolkata", "Personal Care", "Dove Shampoo 200ml", 4500.0, 60, 0.15, 675.0),
    ("B-ORD-1005", "2018-01-25", "B-CUST-001", "Raj Enterprises", "FMCG", "North India", "Delhi", "Beverages", "Pepsi 500ml", 1200.0, 100, 0.0, 240.0),
    ("B-ORD-1006", "2018-02-01", "B-CUST-005", "Pune Distributors", "Wholesale", "West India", "Pune", "Soap & Detergent", "Vim Bar 200g", 900.0, 120, 0.10, 90.0),
    ("B-ORD-1007", "2018-02-05", "B-CUST-006", "Bangalore Hub", "Retail", "South India", "Bangalore", "Snacks", "Kurkure 90g", 1500.0, 75, 0.05, 112.5),
    ("B-ORD-1008", "2018-02-10", "B-CUST-007", "Hyderabad Store", "FMCG", "South India", "Hyderabad", "Personal Care", "Head & Shoulders 180ml", 2800.0, 40, 0.0, 560.0),
    ("B-ORD-1009", "2018-02-15", "B-CUST-002", "Mumbai Traders", "Wholesale", "West India", "Mumbai", "Beverages", "Tropicana 1L", 3600.0, 45, 0.10, 360.0),
    ("B-ORD-1010", "2018-02-20", "B-CUST-008", "Jaipur Retail", "Retail", "North India", "Jaipur", "Soap & Detergent", "Ariel 2kg", 5200.0, 35, 0.05, 780.0),
    ("B-ORD-1011", "2018-03-01", "B-CUST-009", "Ahmedabad Store", "FMCG", "West India", "Ahmedabad", "Snacks", "Bingo 80g", 1100.0, 90, 0.0, 165.0),
    ("B-ORD-1012", "2018-03-05", "B-CUST-010", "Lucknow Mart", "Wholesale", "North India", "Lucknow", "Personal Care", "Pantene 200ml", 3300.0, 55, 0.15, 412.5),
    ("B-ORD-1013", "2018-03-10", "B-CUST-003", "Chennai Retail", "Retail", "South India", "Chennai", "Beverages", "Red Bull 250ml", 4800.0, 25, 0.0, 960.0),
    ("B-ORD-1014", "2018-03-15", None, "Unknown Customer", "FMCG", "North India", "Delhi", "Soap & Detergent", "Lifebuoy 100g", 800.0, 200, 0.0, 160.0),
    ("B-ORD-1015", "2018-03-20", "B-CUST-005", "Pune Distributors", "Wholesale", "West India", "Pune", "Snacks", "Haldirams 200g", 2200.0, 65, 0.10, 220.0),
]

company_b_schema = StructType([
    StructField("order_ref", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("cust_code", StringType(), True),
    StructField("cust_name", StringType(), True),
    StructField("business_type", StringType(), True),
    StructField("zone", StringType(), True),
    StructField("city", StringType(), True),
    StructField("product_category", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("units_sold", IntegerType(), True),
    StructField("discount_rate", DoubleType(), True),
    StructField("net_profit", DoubleType(), True),
])

df_company_b = spark.createDataFrame(company_b_data, schema=company_b_schema)
df_company_b = df_company_b \
    .withColumn("source_company", lit("Company_B")) \
    .withColumn("source_system", lit("Legacy_FMCG_System")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("batch_id", lit("BATCH_001"))

print("=== Company B: Raw Data Loaded ===")
print(f"Rows: {df_company_b.count()} | Columns: {len(df_company_b.columns)}")
df_company_b.show(5)

=== Company B: Raw Data Loaded ===
Rows: 15 | Columns: 17
+----------+----------+----------+---------------+-------------+-----------+-------+----------------+------------------+-------+----------+-------------+----------+--------------+------------------+--------------------+---------+
| order_ref|order_date| cust_code|      cust_name|business_type|       zone|   city|product_category|      product_name|revenue|units_sold|discount_rate|net_profit|source_company|     source_system| ingestion_timestamp| batch_id|
+----------+----------+----------+---------------+-------------+-----------+-------+----------------+------------------+-------+----------+-------------+----------+--------------+------------------+--------------------+---------+
|B-ORD-1001|2018-01-05|B-CUST-001|Raj Enterprises|         FMCG|North India|  Delhi|Soap & Detergent|    Surf Excel 1kg| 2500.0|        50|         0.05|     312.5|     Company_B|Legacy_FMCG_System|2026-07-11 17:15:...|BATCH_001|
|B-ORD-1002|2018-01-10

In [0]:

# Save to Bronze Delta Tables
# Fix Company A column names
df_company_a_fixed = df_company_a \
    .withColumnRenamed("Row ID", "row_id") \
    .withColumnRenamed("Order ID", "order_id") \
    .withColumnRenamed("Order Date", "order_date") \
    .withColumnRenamed("Ship Date", "ship_date") \
    .withColumnRenamed("Ship Mode", "ship_mode") \
    .withColumnRenamed("Customer ID", "customer_id") \
    .withColumnRenamed("Customer Name", "customer_name") \
    .withColumnRenamed("Postal Code", "postal_code") \
    .withColumnRenamed("Product ID", "product_id") \
    .withColumnRenamed("Product Name", "product_name") \
    .withColumnRenamed("Sub-Category", "sub_category")

# Drop existing tables
spark.sql("DROP TABLE IF EXISTS bronze_fmcg.company_a_raw")
spark.sql("DROP TABLE IF EXISTS bronze_fmcg.company_b_raw")

# Save both
df_company_a_fixed.write.format("delta").mode("overwrite").saveAsTable("bronze_fmcg.company_a_raw")
df_company_b.write.format("delta").mode("overwrite").saveAsTable("bronze_fmcg.company_b_raw")

print("=" * 60)
print("BRONZE LAYER: Ingestion Complete!")
print("=" * 60)
print(f"bronze_fmcg.company_a_raw: {spark.read.table('bronze_fmcg.company_a_raw').count()} rows")
print(f"bronze_fmcg.company_b_raw: {spark.read.table('bronze_fmcg.company_b_raw').count()} rows")
print(f"Total Bronze records: {spark.read.table('bronze_fmcg.company_a_raw').count() + spark.read.table('bronze_fmcg.company_b_raw').count()}")

BRONZE LAYER: Ingestion Complete!
bronze_fmcg.company_a_raw: 9994 rows
bronze_fmcg.company_b_raw: 15 rows
Total Bronze records: 10009
